# **1. Install Required Libraries**

In [ ]:
!pip -q install pypdf openai chromadb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 382.9/382.9 kB 27.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 75.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 28.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 121.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 94.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 22.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.7/94.7 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6

## **2. Configure OpenAI API Key**


In [ ]:
import os
from getpass import getpass

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

from openai import OpenAI

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
print("OpenAI client configured successfully.")

Enter your OpenAI API key: ··········
OpenAI client configured successfully.


## **3. Upload the PDF**


In [ ]:
from google.colab import files

uploaded = files.upload()

pdf_path = next(iter(uploaded.keys()))
print("Uploaded file:", pdf_path)

Saving PROITBRIDGE_Employee_Handbook_2026_1.pdf to PROITBRIDGE_Employee_Handbook_2026_1.pdf
Uploaded file: PROITBRIDGE_Employee_Handbook_2026_1.pdf


## **4. PDF Reader**


In [ ]:
from pypdf import PdfReader

reader = PdfReader(pdf_path)

pages = []
for page_number, page in enumerate(reader.pages, start=1):
    page_text = page.extract_text() or ""
    pages.append({
        "page": page_number,
        "text": page_text
    })

text = "\n".join(page["text"] for page in pages)

print("Pages:", len(pages))
print("Characters extracted:", len(text))
print("\nPreview:\n")
print(text[:2000])


Pages: 57
Characters extracted: 104604

Preview:

PEOPLE · POLICIES · PRACTICES
EmployeeHandbook
Everything you need to know about working at
PROITBRIDGE — how we hire, how we pay, how we grow,
how we protect our people and our clients, and what we
expect from each other, every single day.
DOCUMENT Employee Handbook — All Employees
VERSION Version 6.0  |  Effective 1 April 2026
APPLIES TO All full-time, part-time, contract and intern staff in India
OWNER Human Resources Department, PROITBRIDGE
REVIEW CYCLE Annually, or on any change in applicable law
CLASSIFICATION Internal — Confidential
PROITBRIDGE TECHNOLOGIES PRIVATE LIMITED PIB-HR-HB-2026-V6
CONTENTS
Table of Contents
This handbook is organised into 22 chapters. Use the page numbers below to navigate to the policy
you need.
01 Welcome to PROITBRIDGE
A message from the Founder & CEO
4
02 About the Company
Who we are, vision, mission, values, structure
6
03 How to Use This Handbook
Scope, authority, amendments, disclaimer
9
04 Emplo

## **5. Chunking**

For this example, we use a ***simple character-based chunking strategy***. In a production system, chunking can be made more sophisticated using paragraphs, sections, headings, token counts, or semantic boundaries.


In [ ]:
chunk_size = 500
overlap = 50

chunks = []
start = 0
chunk_id = 0

while start < len(text):
    end = start + chunk_size
    chunk_text = text[start:end].strip()

    if chunk_text:
        chunks.append({
            "id": f"chunk-{chunk_id}",
            "text": chunk_text
        })
        chunk_id += 1

    start += chunk_size - overlap

print("Total chunks:", len(chunks))
print("\nFirst chunk:\n")
print(chunks[0]["text"])

Total chunks: 233

First chunk:

PEOPLE · POLICIES · PRACTICES
EmployeeHandbook
Everything you need to know about working at
PROITBRIDGE — how we hire, how we pay, how we grow,
how we protect our people and our clients, and what we
expect from each other, every single day.
DOCUMENT Employee Handbook — All Employees
VERSION Version 6.0  |  Effective 1 April 2026
APPLIES TO All full-time, part-time, contract and intern staff in India
OWNER Human Resources Department, PROITBRIDGE
REVIEW CYCLE Annually, or on any change in applicab


## **6. Generate Embeddings**

Embeddings convert each chunk of text into a numerical vector that represents its semantic meaning.


In [ ]:
EMBEDDING_MODEL = "text-embedding-3-small"

def get_embeddings(texts, batch_size=100):
    all_embeddings = []

    for start in range(0, len(texts), batch_size):
        batch = texts[start:start + batch_size]

        response = client.embeddings.create(model=EMBEDDING_MODEL, input=batch)

        ordered = sorted(response.data, key=lambda item: item.index)
        all_embeddings.extend([item.embedding for item in ordered])

    return all_embeddings

chunk_texts = [chunk["text"] for chunk in chunks]
chunk_embeddings = get_embeddings(chunk_texts)

print("Number of embeddings:", len(chunk_embeddings))
print("Embedding dimensions:", len(chunk_embeddings[0]))
print("First 10 values:", chunk_embeddings[0][:10])

Number of embeddings: 233
Embedding dimensions: 1536
First 10 values: [0.00841522216796875, 0.01490020751953125, 0.051300048828125, 0.02789306640625, -0.0080413818359375, -0.0029754638671875, 0.0248870849609375, 0.0196533203125, -0.023468017578125, 0.002628326416015625]


## **7. Store in the Vector Database**

In [ ]:
import chromadb

chroma_client = chromadb.PersistentClient(path="./chroma_db")
collection = chroma_client.get_or_create_collection(name="employee_handbook")

# Clear the collection so that re-running this notebook does not create duplicates.
existing = collection.get()
if existing["ids"]:
    collection.delete(ids=existing["ids"])

collection.add(ids=[chunk["id"] for chunk in chunks],
               documents=[chunk["text"] for chunk in chunks],
               embeddings=chunk_embeddings,
               metadatas=[{"source": pdf_path} for _ in chunks])

print("Chunks stored in Vector Database:", collection.count())

Chunks stored in Vector Database: 233


## **8. User Query → Query Embedding**

In [ ]:
question = "How many casual leaves are allowed?"

question_embedding = get_embeddings([question])[0]

print("Question:", question)
print("Query embedding dimensions:", len(question_embedding))

Question: How many casual leaves are allowed?
Query embedding dimensions: 1536


## **9. Retriever → Search the Vector Database**

In [ ]:
results

{'ids': [['chunk-70', 'chunk-215', 'chunk-67']],
 'embeddings': None,
 'documents': [['December unless\nleave was refused in writing for business reasons.\nEncashed at basic salary on separation, for the balance standing to your credit.\n7.3 Casual Leave (CL)\nFor short, unplanned personal needs — a bank appointment, a family obligation, a delayed commute.\nMaximum 3 consecutive days at a time. It cannot be combined with EL.\n• \n• \n• \n• \n• \n• \n• \n• \nPROITBRIDGE Employee Handbook\nVersion 6.0 \xa0|\xa0 Effective 1 April 2026 \xa0|\xa0 Internal & Confidential\nPROITBRIDGE | Employee Handbook | Confiden',
   'a long leave count against my balance?\nA. For Earned Leave of 4 days or more, yes — intervening holidays and weekly offs are counted. For Casual\nand Sick Leave, no.\nQ. I forgot to check in for three days last month. Can I fix it?\nA. Regularise within 3 working days in the HRMS. Beyond that, your manager must raise an exception with\nHR, and unresolved days become Loss of 

In [ ]:
results = collection.query(query_embeddings=[question_embedding],n_results=3)

retrieved_chunks = results["documents"][0]
retrieved_ids = results["ids"][0]
distances = results["distances"][0]
print("Retrieved chunks:\n")

for i, (chunk_id, chunk_text, distance) in enumerate(zip(retrieved_ids, retrieved_chunks, distances), start=1):
    print(f"--- Result {i} | {chunk_id} | distance={distance:.4f} ---")
    print(chunk_text[:1000])
    print()

Retrieved chunks:

--- Result 1 | chunk-70 | distance=1.0032 ---
December unless
leave was refused in writing for business reasons.
Encashed at basic salary on separation, for the balance standing to your credit.
7.3 Casual Leave (CL)
For short, unplanned personal needs — a bank appointment, a family obligation, a delayed commute.
Maximum 3 consecutive days at a time. It cannot be combined with EL.
• 
• 
• 
• 
• 
• 
• 
• 
PROITBRIDGE Employee Handbook
Version 6.0  |  Effective 1 April 2026  |  Internal & Confidential
PROITBRIDGE | Employee Handbook | Confiden

--- Result 2 | chunk-215 | distance=1.0518 ---
a long leave count against my balance?
A. For Earned Leave of 4 days or more, yes — intervening holidays and weekly offs are counted. For Casual
and Sick Leave, no.
Q. I forgot to check in for three days last month. Can I fix it?
A. Regularise within 3 working days in the HRMS. Beyond that, your manager must raise an exception with
HR, and unresolved days become Loss of Pay.
Q. Can I

## **10. Prompt + Retrieved Context**

Now we combine the user's question with the relevant chunks returned by the retriever.

This retrieved context becomes part of the input sent to the LLM.


In [ ]:
retrieved_chunks

['December unless\nleave was refused in writing for business reasons.\nEncashed at basic salary on separation, for the balance standing to your credit.\n7.3 Casual Leave (CL)\nFor short, unplanned personal needs — a bank appointment, a family obligation, a delayed commute.\nMaximum 3 consecutive days at a time. It cannot be combined with EL.\n• \n• \n• \n• \n• \n• \n• \n• \nPROITBRIDGE Employee Handbook\nVersion 6.0 \xa0|\xa0 Effective 1 April 2026 \xa0|\xa0 Internal & Confidential\nPROITBRIDGE | Employee Handbook | Confiden',
 'a long leave count against my balance?\nA. For Earned Leave of 4 days or more, yes — intervening holidays and weekly offs are counted. For Casual\nand Sick Leave, no.\nQ. I forgot to check in for three days last month. Can I fix it?\nA. Regularise within 3 working days in the HRMS. Beyond that, your manager must raise an exception with\nHR, and unresolved days become Loss of Pay.\nQ. Can I encash Casual or Sick Leave?\nA. No. Only Earned Leave is encashable, an

In [ ]:
context = "\n\n--- Retrieved Chunk ---\n\n".join(retrieved_chunks)

prompt = f"""Answer the user's question using ONLY the retrieved context below.
             If the answer is not present in the context, say that the information is not available in the provided document.
             Retrieved Context:{context}
             User Question:{question}""".strip()

print(prompt)

Answer the user's question using ONLY the retrieved context below.
             If the answer is not present in the context, say that the information is not available in the provided document.
             Retrieved Context:December unless
leave was refused in writing for business reasons.
Encashed at basic salary on separation, for the balance standing to your credit.
7.3 Casual Leave (CL)
For short, unplanned personal needs — a bank appointment, a family obligation, a delayed commute.
Maximum 3 consecutive days at a time. It cannot be combined with EL.
• 
• 
• 
• 
• 
• 
• 
• 
PROITBRIDGE Employee Handbook
Version 6.0  |  Effective 1 April 2026  |  Internal & Confidential
PROITBRIDGE | Employee Handbook | Confiden

--- Retrieved Chunk ---

a long leave count against my balance?
A. For Earned Leave of 4 days or more, yes — intervening holidays and weekly offs are counted. For Casual
and Sick Leave, no.
Q. I forgot to check in for three days last month. Can I fix it?
A. Regularise withi

## **11. Generate the Answer — OpenAI Responses API**


In [ ]:
GENERATION_MODEL = "gpt-4.1-mini"

response = client.responses.create(model=GENERATION_MODEL,input=prompt)
answer = response.output_text
print("Answer:\n")
print(answer)

Answer:

The number of casual leaves allowed is 8 days, credited upfront on 1 January.
